# Compliance Breach Detection in Access Logs

### Regulatory Compliance Tech Regtech — Banking-AI

Banks must track who accesses sensitive financial data. A compliance breach might occur if an employee accesses customer records at 3 AM or from an unauthorized location.

In this notebook:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of regulatory compliance and RegTech terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Describe how unsupervised anomaly detection works and its relevance to regulatory compliance and RegTech.
- Implement an anomaly detection pipeline on synthetic data and interpret results in operational terms.
- Evaluate the trade-off between detection sensitivity and false-alarm rate in a banking monitoring context.

**Estimated time:** 35–45 minutes

**Collection context:** KYC automation, regulatory reporting, compliance monitoring, bias detection, and data privacy in banking.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** Compliance isn't just about rules; it's about monitoring behavior. Automatically flagging weird log activity helps prevent internal data theft.

**Input data used:** Access logs including Timestamp, User ID, IP Address, Resource Accessed, and Data Volume.

**What we predict:** Anomaly score (Is this access normal or an outlier?).

**ML method used:** Isolation Forest (unsupervised learning that is great for finding "needles in a haystack").

**Learning goal:** Learn how to detect rare events without having pre-labeled data.

**Primary stakeholders:** compliance officers, legal teams, audit managers, and data protection officers.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic Dataset

We'll create 5,000 "normal" logs and 50 "suspicious" ones.

In [ ]:
n_normal = 5000
n_anomalies = 50

# Normal logs: 9 AM to 5 PM, small data transfers
normal_hours = RNG.integers(9, 17, n_normal)
normal_data = RNG.normal(50, 10, n_normal)  # 50 KB average
normal_failed_attempts = RNG.choice([0, 1], n_normal, p=[0.98, 0.02])

# Anomalies: Late night, huge data transfers, or many failed logins
anomaly_hours = RNG.integers(0, 5, n_anomalies)
anomaly_data = RNG.normal(500, 100, n_anomalies)  # 500 KB average
anomaly_failed_attempts = RNG.integers(3, 10, n_anomalies)

df_normal = pd.DataFrame({
    'hour': normal_hours,
    'data_kb': normal_data,
    'failed_logins': normal_failed_attempts,
    'label': 0
})

df_anomaly = pd.DataFrame({
    'hour': anomaly_hours,
    'data_kb': anomaly_data,
    'failed_logins': anomaly_failed_attempts,
    'label': 1
})

df = pd.concat([df_normal, df_anomaly]).sample(frac=1).reset_index(drop=True)
print(f"Generated {len(df)} access logs with {n_anomalies} hidden breaches.")

## Step 4 — Exploratory Data Analysis

EDA

Let's see if we can spot the anomalies visually.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='hour', y='data_kb', hue='label', data=df, alpha=0.6)
plt.title('Log Access Patterns: Time vs Data Volume')
plt.legend(title='Is Breach?')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Scatter plot titled "Log Access Patterns: Time vs Data Volume". The chart highlights spatial separation or clustering patterns in the data.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

In [ ]:
# Features are used as created in Step 3.
print("Target column: 'label'")

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: flag observations beyond a fixed percentile ---
THRESHOLD_PCT = 97
numeric_cols = X.select_dtypes(include='number').columns.tolist()
if 'anomaly_label' in numeric_cols:
    numeric_cols.remove('anomaly_label')
if 'is_anomaly' in numeric_cols:
    numeric_cols.remove('is_anomaly')
thresholds = X[numeric_cols].quantile(THRESHOLD_PCT / 100)
baseline_flags = (X[numeric_cols] > thresholds).any(axis=1)
print(f"Percentile baseline ({THRESHOLD_PCT}th) flags {baseline_flags.sum()} of {len(X)} observations.")
print("Isolation Forest should produce more precise anomaly boundaries.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
X = df.drop('label', axis=1)

# contamination is the expected % of anomalies
model = IsolationForest(contamination=0.015, random_state=RANDOM_SEED)
df['anomaly_pred'] = model.fit_predict(X)

# Isolation Forest returns -1 for anomalies and 1 for normal data
df['anomaly_pred'] = df['anomaly_pred'].map({1: 0, -1: 1})

print("Model flagging complete.")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

print("Classification Report for Breach Detection:")
print(classification_report(df['label'], df['anomaly_pred']))

cm = confusion_matrix(df['label'], df['anomaly_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds')
plt.xlabel('Predicted Anomaly')
plt.ylabel('Actual Breach')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Heatmap with Predicted Anomaly on the x-axis and Actual Breach on the y-axis. The heatmap reveals correlation strength and direction between variables.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

In [ ]:
df['score'] = model.decision_function(X)
print("Logs with the most suspicious scores (lowest values are most anomalous):")
print(df.sort_values('score').head(10))

Why was a specific log flagged? We can look at the "anomaly score".

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: score new observations ---
print("Operational demonstration — anomaly scoring:\n")
new_obs = pd.DataFrame({
    X.columns[0]: [X[X.columns[0]].median(), X[X.columns[0]].quantile(0.99)],
    X.columns[1]: [X[X.columns[1]].median(), X[X.columns[1]].quantile(0.99)],
}, index=["routine", "suspicious"])

scores = model.decision_function(new_obs)
preds  = model.predict(new_obs)
for name, sc, pr in zip(new_obs.index, scores, preds):
    status = "FLAGGED" if pr == -1 else "normal"
    print(f"  {name}: anomaly score = {sc:.3f}  {status}")

print("\nFlagged items would be routed to a human investigator.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end regulatory compliance and RegTech workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Automated compliance systems must be auditable and explainable to regulators.
- AI-driven surveillance can raise employee and customer privacy concerns.
- Bias in compliance models may lead to disproportionate scrutiny of certain groups.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in regulatory compliance and RegTech?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.